In [ ]:
# Definicion de imports de DLT y funciones de Spark
import dlt
from pyspark.sql import functions as F

# Parametros de catalogo y esquema origen
CATALOGO = "workspace"
ESQUEMA = "ingenieria_datos_tunombre"
TABLA_BRONZE = f"{CATALOGO}.{ESQUEMA}.ventas_bronze"


In [ ]:
# Definicion de la vista materializada pipeline_ventas_limpias
@dlt.table(
    name="pipeline_ventas_limpias",
    comment="Ventas limpias con deduplicacion y reglas de calidad aplicadas"
)
def pipeline_ventas_limpias():
    # Lectura de la tabla bronze como fuente
    df_bronze = spark.read.table(TABLA_BRONZE)
    
    # Eliminacion de duplicados exactos
    df_sin_duplicados = df_bronze.dropDuplicates()
    
    # Filtro de calidad: producto existente, cantidad positiva y precio positivo
    df_filtrado = df_sin_duplicados.filter(
        F.col("producto").isNotNull() &
        (F.trim(F.col("producto")) != "") &
        (F.col("cantidad") > 0) &
        (F.col("precio_unitario") > 0)
    )
    
    # Calculo del total de venta
    return df_filtrado.withColumn(
        "total_venta",
        F.round(F.col("cantidad") * F.col("precio_unitario"), 2)
    )


In [ ]:
# Definicion de la vista materializada dependiente pipeline_ventas_por_categoria
@dlt.table(
    name="pipeline_ventas_por_categoria",
    comment="Resumen de total vendido por categoria dependiente de pipeline_ventas_limpias"
)
def pipeline_ventas_por_categoria():
    # Lectura declarativa del conjunto de datos procesado en el mismo pipeline
    return (
        dlt.read("pipeline_ventas_limpias")
        .groupBy("categoria")
        .agg(F.round(F.sum("total_venta"), 2).alias("total_vendido"))
    )
